In [3]:
import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold

/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


In [ ]:
# 1. PREPROCESSING FUNCTION 
def preprocess_features(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df['sale_year'] = df['date'].dt.year
    df['sale_month'] = df['date'].dt.month
    df['day_of_month'] = df['date'].dt.day
    df['day_of_week'] = df['date'].dt.dayofweek

    # Feature Engineering
    # Vectorized (MUCH faster than apply)
    df['effective_built_year'] = np.where(
        df['yr_renovated'] > 0,
        df['yr_renovated'],
        df['yr_built']
    )
    df['house_age'] = df['sale_year'] - df['effective_built_year']

    return df.drop(columns=['id', 'date'], errors='ignore') 


In [5]:
# --- 2. LOAD & PREPARE ---
train_df = pd.read_csv('/kaggle/input/property-val-dataset/train.csv')
test_df = pd.read_csv('/kaggle/input/property-val-dataset/test.csv')

X = preprocess_features(train_df)
y = np.log1p(train_df['price'])
X = X.drop(columns=['price'])

# Convert once
dtrain = xgb.DMatrix(X.values, label=y.values)